# Data cleaning

The cleaning itself lives in `coffee.clean`, so it is tested and runs in CI.
This notebook loads the raw layer, runs it, and eyeballs the result.

To regenerate the cleaned layer without opening Jupyter:

```bash
uv run clean-reviews
```

- [Load](#load)
- [Clean](#clean)
- [Check](#check)
- [Export](#export)

## Load <a id='load'></a>

In [ ]:
%reload_ext autoreload
%autoreload 2


import matplotlib as mpl
import pandas as pd

from coffee.clean import clean_reviews
from coffee.config import DATA_DIR
from coffee.enrich import enrich_reviews, load_cpi, load_exchange_rates

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 100)
mpl.rcParams["figure.dpi"] = 300

In [ ]:
raw: pd.DataFrame = pd.read_csv(DATA_DIR / "raw" / "reviews.csv")

crosswalk_path = DATA_DIR / "roasters" / "roaster_crosswalk.csv"
crosswalk = pd.read_csv(crosswalk_path) if crosswalk_path.exists() else None

print(f"{len(raw)} raw reviews")
raw.info()

## Clean <a id='clean'></a>

One call. Each step is a public function in `coffee.clean` if you want to run
them one at a time while exploring.

In [ ]:
# The cleaned layer depends on the raw scrape and nothing else.
df = clean_reviews(raw, crosswalk=crosswalk)

dropped = len(raw) - len(df)
print(f"{len(raw)} raw -> {len(df)} cleaned ({dropped} agtron typos dropped)")
df.info()

## Check <a id='check'></a>

In [ ]:
# Fail fast if the pipeline regresses.
assert df["rating"].dropna().between(0, 100).all(), "rating outside 0-100"
assert (df["quantity_in_lbs"].dropna() > 0).all(), "non-positive quantity_in_lbs"
assert df["url"].is_unique, "duplicate reviews"

df.isna().sum().sort_values(ascending=False).head(20)

In [ ]:
# Comparing prices across currencies and decades needs reference data the
# reviews do not carry, so it happens after the cleaned layer rather than
# inside it. Kept in memory here; nothing below is written to data/clean.
priced = enrich_reviews(
    df,
    exchange_rates=load_exchange_rates(
        DATA_DIR / "external" / "openex_exchange_rates.json"
    ),
    cpi=load_cpi(DATA_DIR / "external" / "consumer_price_index.csv"),
)

priced.groupby("price_currency")[["price_value", "price_usd", "price_usd_adj"]].median()

In [ ]:
priced.groupby(priced["review_date"].dt.year)["price_usd_adj_per_lb"].median().tail(15)

In [ ]:
display(df["quantity_unit"].value_counts())
display(df["roaster_country"].value_counts().head(10))
display(df["origin_country"].value_counts().head(10))

## Export <a id='export'></a>

In [ ]:
out_path = DATA_DIR / "clean" / "reviews.csv"
df.to_csv(out_path, index=False)
print(f"wrote {len(df)} rows to {out_path}")